# Module 3.2: Memory Identification

In Module 2 (Episodic Memory), we gave the agent a `remember_event` tool and told it
to store important facts. But we left a critical question unanswered:

> **How does the agent decide what's worth remembering?**

Without explicit criteria, agents either store too much (noise, hypotheticals, sensitive data)
or too little (missing key preferences). This notebook builds the identification layer
that sits between raw conversation and memory storage.

In [1]:
%pip install -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, os, json
import sniffio

sys.path.insert(0, "..")
sniffio.current_async_library_cvar.set("asyncio")

from shared.travel_agent import create_client, SYSTEM_PROMPT
from lifecycle_utils import MemoryCandidate, MemoryDecision, MemoryItem, MemoryState

client, credential = create_client("../.env")
print("Client ready")

c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_skills.py:122: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Users\divyesheth\OneDrive - Microsoft\Documents\python-projects\agentic-memory\.venv\Lib\site-packages\agent_framework\_harness\_file_access.py:602: ExperimentalWarning: [HARNESS] AgentFileStore is experimental and may change or be removed in future versions without notice.


Client ready


## The Problem: Storing Everything

What happens when an agent simply stores every user turn as a "memory"?
Let's simulate a 10-turn conversation and blindly persist all of it.

In [3]:
# A realistic 10-turn conversation between Sarah and the travel agent
SAMPLE_CONVERSATION = [
    # Turn 1: Durable preference (should memorise)
    "I always prefer window seats on long flights — I like watching the landscape.",
    # Turn 2: Transient request (should discard)
    "Can you check if there's a flight to Chicago tomorrow?",
    # Turn 3: Durable fact (should memorise)
    "I'm based in San Francisco, so SFO is my home airport.",
    # Turn 4: Hypothetical (should discard)
    "What if I wanted to fly first class — how much more would that be?",
    # Turn 5: Strong preference (should memorise)
    "I really don't like layovers longer than 2 hours. I'd rather pay more for direct.",
    # Turn 6: Other person's info (should discard)
    "My colleague Mike says the Hilton downtown is terrible.",
    # Turn 7: Actionable constraint (should memorise)
    "My company reimburses up to $250/night for hotels, so keep it under that.",
    # Turn 8: Session-specific (should discard)
    "Actually, go back to that first option you showed me.",
    # Turn 9: Repeated preference confirmation (should memorise)
    "Yes, Marriott is my go-to chain. I have their loyalty program.",
    # Turn 10: Sensitive data (should discard)
    "My loyalty number is MR-998877-2024.",
]

# ─── Naive approach: store EVERYTHING as memory ───────────────────────────────
naive_memory_store = []

for i, msg in enumerate(SAMPLE_CONVERSATION, 1):
    naive_memory_store.append({
        "id": i,
        "content": msg,
        "stored_at": "2026-07-09",
    })

print(f"Naive store: {len(naive_memory_store)} memories (1 per turn)\n")
for m in naive_memory_store:
    print(f"  Memory {m['id']:2d}: {m['content'][:70]}{'...' if len(m['content']) > 70 else ''}")

Naive store: 10 memories (1 per turn)

  Memory  1: I always prefer window seats on long flights — I like watching the lan...
  Memory  2: Can you check if there's a flight to Chicago tomorrow?
  Memory  3: I'm based in San Francisco, so SFO is my home airport.
  Memory  4: What if I wanted to fly first class — how much more would that be?
  Memory  5: I really don't like layovers longer than 2 hours. I'd rather pay more ...
  Memory  6: My colleague Mike says the Hilton downtown is terrible.
  Memory  7: My company reimburses up to $250/night for hotels, so keep it under th...
  Memory  8: Actually, go back to that first option you showed me.
  Memory  9: Yes, Marriott is my go-to chain. I have their loyalty program.
  Memory 10: My loyalty number is MR-998877-2024.


Now when the agent tries to retrieve relevant memories for a new conversation,
the noise overwhelms the signal:

In [4]:
# Simulate: agent queries "What do I know about this user's hotel preferences?"
# With naive storage, a keyword match returns ALL of this:
query_keywords = ["hotel", "marriott", "hilton", "night", "loyalty"]
retrieved = [m for m in naive_memory_store
             if any(k in m["content"].lower() for k in query_keywords)]

print(f"Query: 'hotel preferences for Sarah'\n")
print(f"Retrieved {len(retrieved)} memories:\n")
for m in retrieved:
    print(f"  Memory {m['id']:2d}: {m['content']}")
    
print("\n" + "=" * 70)
print("PROBLEMS with these results:")
print("  • Memory  6: Someone ELSE's opinion (Mike's, not Sarah's)")
print("  • Memory 10: Sensitive loyalty number (privacy violation!)")
print("  • Memory  7: Useful — but mixed in with noise")
print("  • Memory  9: Useful — but how do we distinguish from the rest?")

Query: 'hotel preferences for Sarah'

Retrieved 4 memories:

  Memory  6: My colleague Mike says the Hilton downtown is terrible.
  Memory  7: My company reimburses up to $250/night for hotels, so keep it under that.
  Memory  9: Yes, Marriott is my go-to chain. I have their loyalty program.
  Memory 10: My loyalty number is MR-998877-2024.

PROBLEMS with these results:
  • Memory  6: Someone ELSE's opinion (Mike's, not Sarah's)
  • Memory 10: Sensitive loyalty number (privacy violation!)
  • Memory  7: Useful — but mixed in with noise
  • Memory  9: Useful — but how do we distinguish from the rest?


## What Went Wrong

Storing every user turn as memory creates four distinct failure modes:

| Failure | Example from our conversation | Impact |
|---------|-------------------------------|--------|
| **Session noise** | "Go back to that first option" (turn 8) | Meaningless in future sessions |
| **Hypotheticals** | "What if I wanted first class" (turn 4) | Agent thinks user wants first class |
| **Wrong attribution** | "My colleague Mike says…" (turn 6) | Agent stores someone else's opinion as Sarah's |
| **Sensitive data** | "My loyalty number is…" (turn 10) | Privacy violation — shouldn't persist |

The core issue: **not every piece of information deserves to be remembered**.
We need explicit criteria to separate signal from noise *before* storage.

## The Solution: Memory Identification Criteria

We score every candidate memory on four dimensions:

| Criterion | Definition | Example (High) | Example (Low) |
|-----------|------------|----------------|---------------|
| **Durability** | How likely to remain true over weeks/months | "I'm vegetarian" | "I'm in a rush today" |
| **Reusability** | How likely to be useful in future interactions | "Prefers aisle seats" | "Flight UA123 departs at 3pm" |
| **User-Specificity** | How personal vs generic knowledge | "Sarah hates layovers" | "JFK has 6 terminals" |
| **Actionability** | Can the agent use this to improve service | "Budget max $300/night" | "Weather is nice today" |

A composite score (average of all four) determines the decision:
- **≥ 0.6** → Memorise
- **0.4 – 0.6** → Ask the user for confirmation
- **< 0.4** → Discard

### Reference

> *"Towards Root Memories"* (arXiv:2606.23283) — introduces the distinction between
> reusable, decision-affecting "root" memories vs noise. Our identification criteria
> are inspired by their characterisation of memories that change downstream behaviour.

## Building the Memory Identifier

We use the LLM itself as the classifier. Given a user turn, it scores each criterion
0.0–1.0 and returns a structured JSON decision. This keeps the logic transparent
and tunable (change the prompt, not code).

In [5]:
IDENTIFICATION_PROMPT = """You are a memory identification system. Given a conversation turn from the user,
determine whether it contains information worth memorising for future interactions.

Score each criterion 0.0 to 1.0:
- durability: Will this likely remain true for weeks/months?
- reusability: Will this be useful in future conversations?
- user_specificity: Is this personal to this user (not generic knowledge)?
- actionability: Can the agent use this to improve service?

Categories: preference | fact | event | procedure

Decision rules:
- If average score >= 0.6 → memorise
- If average score >= 0.4 and < 0.6 → ask_user
- Otherwise → discard

ANTI-PATTERNS (always discard):
- One-off questions ("what time is it")
- Hypotheticals ("what if I went to Paris")
- Other people's information ("my colleague likes sushi")
- Session-specific context ("as I said earlier")
- Sensitive data that shouldn't persist (credit card numbers, passwords, loyalty numbers)

Respond with ONLY valid JSON:
{
  "content": "extracted fact to store",
  "category": "preference|fact|event|procedure",
  "durability": 0.0,
  "reusability": 0.0,
  "user_specificity": 0.0,
  "actionability": 0.0,
  "decision": "memorise|discard|ask_user",
  "reasoning": "brief explanation"
}"""

print("Identification prompt defined")
print(f"  Anti-patterns: hypotheticals, others' info, sensitive data, session-specific")

Identification prompt defined
  Anti-patterns: hypotheticals, others' info, sensitive data, session-specific


The `MemoryIdentifier` class wraps the LLM classification into a reusable component.
It takes a user message, calls the LLM with the identification prompt, and returns
a structured `MemoryCandidate` with scores.

In [6]:
from agent_framework._types import Message

class MemoryIdentifier:
    """LLM-based classifier that determines if a user turn contains memorable info."""

    def __init__(self, client, threshold: float = 0.6, ask_threshold: float = 0.4):
        self.client = client
        self.threshold = threshold
        self.ask_threshold = ask_threshold

    async def evaluate(self, user_message: str) -> MemoryCandidate:
        """Evaluate a single user message for memorability."""
        messages = [
            Message(role="system", contents=[IDENTIFICATION_PROMPT]),
            Message(role="user", contents=[user_message]),
        ]
        response = await self.client.get_response(messages=messages)
        raw = response.text.strip()
        # Strip markdown code fences if present
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[1].rsplit("```", 1)[0]
        data = json.loads(raw)
        return MemoryCandidate(
            content=data["content"],
            category=data["category"],
            durability=data["durability"],
            reusability=data["reusability"],
            user_specificity=data["user_specificity"],
            actionability=data["actionability"],
            decision=MemoryDecision(data["decision"]),
            reasoning=data["reasoning"],
        )

    async def evaluate_conversation(self, messages: list[str]) -> list[MemoryCandidate]:
        """Evaluate all user turns in a conversation."""
        candidates = []
        for msg in messages:
            candidate = await self.evaluate(msg)
            candidates.append(candidate)
        return candidates


identifier = MemoryIdentifier(client)
print("MemoryIdentifier ready")

MemoryIdentifier ready


## Demo: Same Conversation, With Identification

Let's run the same 10-turn conversation through the identifier. Compare the output
to the naive "store everything" approach above — the classifier should keep ~4–5
turns and discard the rest.

In [7]:
print(f"Evaluating {len(SAMPLE_CONVERSATION)} conversation turns...\n")

candidates = await identifier.evaluate_conversation(SAMPLE_CONVERSATION)

for i, (msg, candidate) in enumerate(zip(SAMPLE_CONVERSATION, candidates), 1):
    icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[candidate.decision.value]
    print(f"Turn {i:2d} {icon} [{candidate.decision.value:8s}] "
          f"(score={candidate.composite_score:.2f})")
    print(f"         User: {msg[:65]}{'...' if len(msg) > 65 else ''}")
    if candidate.decision != MemoryDecision.DISCARD:
        print(f"         → Memory: {candidate.content}")
        print(f"           Category: {candidate.category} | {candidate.reasoning}")
    print()

Evaluating 10 conversation turns...

Turn  1 ✅ [memorise] (score=0.92)
         User: I always prefer window seats on long flights — I like watching th...
         → Memory: Prefers window seats on long flights (likes watching the landscape).
           Category: preference | This is a stable personal preference useful for future travel-related suggestions and seat recommendations.

Turn  2 ❌ [discard ] (score=0.00)
         User: Can you check if there's a flight to Chicago tomorrow?

Turn  3 ✅ [memorise] (score=0.91)
         User: I'm based in San Francisco, so SFO is my home airport.
         → Memory: User is based in San Francisco; SFO is their home airport.
           Category: fact | Personal, durable location detail useful for future travel-related suggestions and tailoring; high likelihood of remaining true for months.

Turn  4 ❌ [discard ] (score=0.00)
         User: What if I wanted to fly first class — how much more would that be...

Turn  5 ✅ [memorise] (score=0.90)
     

## Score Distribution

The composite scores should show a clear separation between memorable and transient
information — the threshold acts as a decision boundary.

In [8]:
print("Composite Score by Turn:")
print("=" * 60)
for i, c in enumerate(candidates, 1):
    bar = "█" * int(c.composite_score * 40)
    icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[c.decision.value]
    print(f"Turn {i:2d} {icon} |{bar:<40}| {c.composite_score:.2f}")

memorised = sum(1 for c in candidates if c.decision == MemoryDecision.MEMORISE)
discarded = sum(1 for c in candidates if c.decision == MemoryDecision.DISCARD)
ask_user = sum(1 for c in candidates if c.decision == MemoryDecision.ASK_USER)

print("\n" + "=" * 60)
print(f"Threshold: memorise ≥ 0.6 | ask_user ≥ 0.4 | discard < 0.4")
print(f"\n✅ Memorise: {memorised}  |  ❌ Discard: {discarded}  |  ❓ Ask user: {ask_user}")
print(f"\nComparison:")
print(f"  Naive store-everything:  10 memories (including noise, sensitive data)")
print(f"  With identification:     {memorised} memories (clean, actionable, durable)")

Composite Score by Turn:
Turn  1 ✅ |█████████████████████████████████████   | 0.92
Turn  2 ❌ |                                        | 0.00
Turn  3 ✅ |████████████████████████████████████    | 0.91
Turn  4 ❌ |                                        | 0.00
Turn  5 ✅ |████████████████████████████████████    | 0.90
Turn  6 ❌ |███                                     | 0.08
Turn  7 ✅ |███████████████████████████████████     | 0.88
Turn  8 ❌ |                                        | 0.00
Turn  9 ✅ |███████████████████████████████████     | 0.88
Turn 10 ❌ |█████████████████████████████████████   | 0.92

Threshold: memorise ≥ 0.6 | ask_user ≥ 0.4 | discard < 0.4

✅ Memorise: 5  |  ❌ Discard: 5  |  ❓ Ask user: 0

Comparison:
  Naive store-everything:  10 memories (including noise, sensitive data)
  With identification:     5 memories (clean, actionable, durable)


## Edge Cases: What NOT to Memorise

The classifier's value is in its ability to handle tricky inputs that *look* like
preferences but should NOT be stored.

In [9]:
EDGE_CASES = [
    # Looks like preference but is hypothetical
    "If I were to move to London, I'd probably want to fly British Airways.",
    # Looks like a fact but is about someone else
    "My boss always flies Delta — maybe I should try them too.",
    # Contains sensitive data that shouldn't persist
    "My passport number is AB1234567, expiring March 2028.",
    # Temporary state, not durable
    "I'm feeling sick today so I might cancel my trip.",
    # Genuine durable preference (control case — should memorise)
    "I have a severe peanut allergy — please always flag this for meal selection.",
]

print("Edge Case Classification:\n")
for msg in EDGE_CASES:
    candidate = await identifier.evaluate(msg)
    icon = {"memorise": "✅", "discard": "❌", "ask_user": "❓"}[candidate.decision.value]
    display = f'"{msg[:65]}..."' if len(msg) > 65 else f'"{msg}"'
    print(f"{icon} [{candidate.decision.value:8s}] {display}")
    print(f"   Score: {candidate.composite_score:.2f} | {candidate.reasoning}\n")

Edge Case Classification:

✅ [memorise] "If I were to move to London, I'd probably want to fly British Air..."
   Score: 0.82 | User stated a personal travel preference that is likely to remain useful for future travel recommendations or planning.

❌ [discard ] "My boss always flies Delta — maybe I should try them too."
   Score: 0.30 | This is information about another person (the user's boss), which falls into the 'other people's information' anti-pattern and should not be stored.

❌ [discard ] "My passport number is AB1234567, expiring March 2028."
   Score: 0.50 | Contains sensitive personal data (passport number); per anti-patterns and privacy policy, such information must not be stored.

❌ [discard ] "I'm feeling sick today so I might cancel my trip."
   Score: 0.35 | This is a transient health status and a possible immediate action; it's personal but unlikely to remain true or be useful long-term, so it should not be memorised.

✅ [memorise] "I have a severe peanut allergy — ple

## Production Hardening: Filter Pipeline

In production, identified candidates pass through additional filters before storage:

1. **Category allowlist** — Only store certain categories (e.g., no "event" for privacy)
2. **Minimum confidence** — Below threshold, discard even if classified as "memorise"
3. **Rate limiting** — Don't store more than N memories per conversation

In [10]:
from dataclasses import dataclass

@dataclass
class FilterConfig:
    allowed_categories: list = None
    min_composite_score: float = 0.5
    max_memories_per_conversation: int = 10

    def __post_init__(self):
        if self.allowed_categories is None:
            self.allowed_categories = ["preference", "fact", "procedure"]


class MemoryFilterPipeline:
    """Post-identification filtering before memory storage."""

    def __init__(self, config: FilterConfig = None):
        self.config = config or FilterConfig()

    def apply(self, candidates: list[MemoryCandidate]) -> list[MemoryCandidate]:
        """Filter candidates through the pipeline."""
        # Step 1: Only keep 'memorise' decisions
        kept = [c for c in candidates if c.decision == MemoryDecision.MEMORISE]

        # Step 2: Category allowlist
        kept = [c for c in kept if c.category in self.config.allowed_categories]

        # Step 3: Minimum score
        kept = [c for c in kept if c.composite_score >= self.config.min_composite_score]

        # Step 4: Rate limit (keep highest-scored)
        kept.sort(key=lambda c: c.composite_score, reverse=True)
        kept = kept[: self.config.max_memories_per_conversation]

        return kept


pipeline = MemoryFilterPipeline()
filtered = pipeline.apply(candidates)

print(f"Before filtering: {len(candidates)} candidates")
print(f"After filtering:  {len(filtered)} memories to store\n")
for c in filtered:
    print(f"  [{c.category:10s}] {c.content}")
    print(f"              Score: {c.composite_score:.2f} | {c.reasoning}\n")

Before filtering: 10 candidates
After filtering:  5 memories to store

  [preference] Prefers window seats on long flights (likes watching the landscape).
              Score: 0.92 | This is a stable personal preference useful for future travel-related suggestions and seat recommendations.

  [fact      ] User is based in San Francisco; SFO is their home airport.
              Score: 0.91 | Personal, durable location detail useful for future travel-related suggestions and tailoring; high likelihood of remaining true for months.

  [preference] Prefers direct flights; dislikes layovers longer than 2 hours and is willing to pay more to avoid them.
              Score: 0.90 | This is a stable personal travel preference that will be useful for future trip planning and can be acted on by prioritising direct flights or avoiding layovers >2 hours.

  [fact      ] User's company reimburses up to $250 per night for hotels; suggest stays under $250/night.
              Score: 0.88 | This is a st

## Connecting to the Lifecycle

Identified memories don't go directly to "trusted" status — they enter the lifecycle
as **candidates**. This is the handoff to Notebook 03 (Staged Promotion):

```
User message → MemoryIdentifier → MemoryCandidate → FilterPipeline → MemoryItem(state=CANDIDATE)
                                                                          ↓
                                                              Notebook 03: Staged Promotion
```

In [11]:
def candidate_to_memory(candidate: MemoryCandidate, user_id: str) -> MemoryItem:
    """Convert an identified candidate into a lifecycle-managed MemoryItem."""
    return MemoryItem(
        user_id=user_id,
        content=candidate.content,
        category=candidate.category,
        state=MemoryState.CANDIDATE,  # Always starts as candidate!
        confidence=candidate.composite_score,
        source_type="llm_inference",
    )

# Convert our filtered candidates to MemoryItems
memory_items = [candidate_to_memory(c, "E001") for c in filtered]

print(f"Created {len(memory_items)} MemoryItems (all in CANDIDATE state):\n")
for m in memory_items:
    print(f"  [{m.state.value:10s}] {m.content}")
    print(f"              confidence={m.confidence:.2f} | category={m.category}")
    print(f"              → Needs confirmation before agent uses this\n")

Created 5 MemoryItems (all in CANDIDATE state):

  [candidate ] Prefers window seats on long flights (likes watching the landscape).
              confidence=0.92 | category=preference
              → Needs confirmation before agent uses this

  [candidate ] User is based in San Francisco; SFO is their home airport.
              confidence=0.91 | category=fact
              → Needs confirmation before agent uses this

  [candidate ] Prefers direct flights; dislikes layovers longer than 2 hours and is willing to pay more to avoid them.
              confidence=0.90 | category=preference
              → Needs confirmation before agent uses this

  [candidate ] User's company reimburses up to $250 per night for hotels; suggest stays under $250/night.
              confidence=0.88 | category=fact
              → Needs confirmation before agent uses this

  [candidate ] Prefers Marriott as go-to hotel chain and is a member of the Marriott loyalty program.
              confidence=0.88 | ca

## Key Takeaways

1. **Not everything is memory** — without filtering, agents store noise, hypotheticals, and sensitive data
2. **Four dimensions** — durability, reusability, user-specificity, actionability
3. **Anti-patterns matter** — hypotheticals, others' info, sensitive data must be excluded
4. **Pipeline filtering** — category allowlist, score threshold, rate limiting
5. **Memory starts as candidate** — identification doesn't equal trust (→ Notebook 03)

## Next: Staged Promotion (Notebook 03)

Now that we can identify *what* to store, the next question is:
**when should the agent trust it?** Newly identified memories must earn confidence
through repeated confirmation before influencing agent behaviour.